# p138 Copy List with Random Pointer 解析笔记

- 题号：p138
- 题目英文名：Copy List with Random Pointer
- 题目中文名：复制带随机指针的链表
- 题目链接：https://leetcode.cn/problems/copy-list-with-random-pointer/
- 题型：链表 / 深拷贝
- 难度：Medium
- 推荐优先级：中
- Java 原代码位置：`solutions-java/leetcode/p138_copy_list_with_random_pointer/CopyListWithRandomPointer.java`


## 1. 题目一句话总结

这道题要求我们深拷贝一条带 `next` 和 `random` 两类指针的链表，既要复制节点值，也要正确复制所有引用关系。

本质上考的是“旧节点如何映射到新节点”，以及如何在不借助哈希表的情况下完成这种映射。


## 2. 题目理解与约束分析

### 2.1 题目要求

给你一条特殊链表，每个节点除了有正常的 `next` 指针外，还有一个 `random` 指针，它可以指向链表中的任意节点，也可以指向 `None`。

我们需要返回一条全新的链表，使得：

- 新链表的节点值与原链表相同
- `next` 关系一致
- `random` 关系一致
- 新链表中的指针不能再指向旧链表节点

### 2.2 输入与输出

- 输入：原链表头节点 `head`
- 输出：深拷贝后的新链表头节点
- 返回结果含义：返回完整复制后的独立链表

### 2.3 关键约束

- 链表可能为空
- `random` 可能为 `None`
- `random` 可能指向任意位置，甚至指向自己
- 题目更进阶的写法希望做到 `O(1)` 额外空间

### 2.4 示例分析

假设链表序列化后是：`[(7, None), (13, 0), (11, 1)]`

其中元组 `(val, random_index)` 的含义是：

- 第一个节点值为 `7`，`random = None`
- 第二个节点值为 `13`，`random` 指向下标 `0` 的节点
- 第三个节点值为 `11`，`random` 指向下标 `1` 的节点

复制后的链表序列化结果也应该完全相同，但所有节点对象必须是新的对象。


## 3. Java 原代码

```java
package leetcode.p138_copy_list_with_random_pointer;

public class CopyListWithRandomPointer {

    public static class Node {
        public int val;
        public Node next;
        public Node random;

        public Node(int v) {
            val = v;
        }
    }

    public static Node copyRandomList(Node head) {
        if (head == null) {
            return null;
        }

        Node cur = head;
        Node next = null;
        while (cur != null) {
            next = cur.next;
            cur.next = new Node(cur.val);
            cur.next.next = next;
            cur = next;
        }

        cur = head;
        Node copy = null;
        while (cur != null) {
            next = cur.next.next;
            copy = cur.next;
            if (cur.random != null) {
                copy.random = cur.random.next;
            } else {
                copy.random = null;
            }
            cur = next;
        }

        Node result = head.next;

        cur = head;
        while (cur != null) {
            next = cur.next.next;
            copy = cur.next;
            cur.next = next;
            copy.next = next != null ? next.next : null;
            cur = next;
        }

        return result;
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案没有用哈希表，而是用了这题最经典的“节点穿插”技巧。

它把整道题拆成 3 次遍历：

1. 复制每个节点，并把复制节点插到原节点后面
2. 利用这种穿插结构设置复制节点的 `random`
3. 再把新旧链表拆开

为什么这个方案聪明？因为一旦把链表改造成：

```text
A -> A' -> B -> B' -> C -> C'
```

那么：

- 原节点 `A` 的复制节点就是 `A.next`
- 原节点 `A.random` 指向的那个旧节点，它的复制节点就是 `A.random.next`

这样就相当于把“旧节点 -> 新节点”的映射藏在链表结构本身里，不需要额外哈希表。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最容易想到的方案是哈希表：

1. 第一遍复制节点，并记录 `旧节点 -> 新节点`
2. 第二遍根据哈希表补 `next` 和 `random`

### 5.2 为什么还能继续优化

哈希表方案时间复杂度已经是 `O(n)`，但是额外空间也是 `O(n)`。

Java 原方案进一步优化的方向，是把这份映射直接编码到链表结构里，避免额外容器。

### 5.3 优化方向

通过穿插复制节点：

- 让每个旧节点旁边就紧跟自己的新节点
- 从而把 `random` 的映射也转成常数时间可达
- 最后再把新旧两条链拆开

于是整体能做到：

- 时间复杂度 `O(n)`
- 额外空间复杂度 `O(1)`


## 6. 核心算法讲解

### 6.1 算法名称

- 链表节点穿插复制
- 原地映射

### 6.2 为什么想到这个算法

因为这题最难的不是复制值，而是复制引用关系。特别是 `random` 这种“跳着指”的关系，如果没有映射，就很难在常数时间内找到对应的新节点。

把复制节点直接插在原节点后面后，映射关系天然成立：

- 原节点的复制节点：`cur.next`
- 原节点随机目标的复制节点：`cur.random.next`

### 6.3 关键状态

- `cur`：当前遍历到的原链表节点
- `next_node`：缓存下一个旧节点
- `copy`：当前旧节点对应的新节点
- `result` / `new_head`：新链表头节点

### 6.4 步骤拆解

1. 第一遍遍历，把每个复制节点插到原节点后面
2. 第二遍遍历，根据 `cur.random.next` 设置复制节点的 `random`
3. 记录新链表头 `new_head = head.next`
4. 第三遍遍历，把交错的新旧链表拆成两条独立链表
5. 返回新链表头节点


## 7. 过程演示

设原链表是：

```text
A(7) -> B(13) -> C(11)
B.random -> A
C.random -> B
```

### 7.1 第一步：复制并穿插

```text
A -> A' -> B -> B' -> C -> C'
```

### 7.2 第二步：设置 random

此时：

- `A' = A.next`
- `B' = B.next`
- `C' = C.next`

如果 `B.random = A`，那么 `B'.random = A' = A.next`

也就是：

```python
copy.random = cur.random.next if cur.random else None
```

### 7.3 第三步：拆链

```text
旧链表：A -> B -> C
新链表：A' -> B' -> C'
```

这一步要同时完成两件事：

- 恢复旧链表
- 串起新链表


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(eq=False)
class Node:
    val: int
    next: Node | None = None
    random: Node | None = None


In [ ]:
class Solution:
    def copyRandomList(self, head: Node | None) -> Node | None:
        if head is None:
            return None

        cur = head
        while cur is not None:
            next_node = cur.next
            cur.next = Node(cur.val)
            cur.next.next = next_node
            cur = next_node

        cur = head
        while cur is not None:
            copy = cur.next
            copy.random = cur.random.next if cur.random is not None else None
            cur = copy.next

        new_head = head.next
        cur = head
        while cur is not None:
            copy = cur.next
            next_node = copy.next

            cur.next = next_node
            copy.next = next_node.next if next_node is not None else None
            cur = next_node

        return new_head


## 8. 代码逐段讲解

### 8.1 第一次遍历

第一次遍历只做一件事：把复制节点插到原节点后面。执行完后，链表会从：

```text
A -> B -> C
```

变成：

```text
A -> A' -> B -> B' -> C -> C'
```

### 8.2 第二次遍历

第二次遍历利用穿插结构补 `random`。因为复制节点总在原节点后面，所以新节点的随机目标也总能通过 `.next` 找到。

### 8.3 第三次遍历

第三次遍历的难点是同时维护两条链：

- `cur.next = next_node`：恢复旧链表
- `copy.next = next_node.next if next_node else None`：串起新链表

### 8.4 返回结果

`new_head = head.next` 在第一次穿插完成后就已经确定了，它就是整条复制链表的头节点。


## 9. 复杂度分析

- 时间复杂度：`O(n)`
- 为什么是这个时间复杂度：一共做 3 次线性遍历，每个节点只做常数次指针修改
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：没有使用与节点数量成比例的额外辅助结构，输出链表本身不计入额外空间


## 10. 易错点

1. 直接写 `copy.random = cur.random`，这样会让新链表的 `random` 指回旧链表。
2. 第一遍穿插后，第二遍遍历时忘了按“跳两个节点”的节奏前进。
3. 第三遍拆链时没有先缓存 `next_node`，导致链表断掉后丢失后续节点。
4. 忘记处理空链表，访问 `head.next` 时会报错。
5. 误以为这题必须用哈希表，其实 Java 原方案已经把映射关系内嵌到链表结构里了。


## 11. 面试时怎么讲

可以这样概括：

> 最直接的方法是用哈希表维护旧节点到新节点的映射，但我这里会用更省空间的穿插复制法。
> 我先把每个复制节点插到原节点后面，这样旧节点和新节点天然配对。
> 接着利用 `cur.random.next` 给复制节点设置 `random`。
> 最后再把交错的新旧链表拆开，恢复旧链表并拿到新链表。
> 这样时间复杂度是 `O(n)`，额外空间复杂度是 `O(1)`。


## 12. 实际应用场景

这题可以类比为：复制一条带主连接和跨节点引用的对象链，同时保持所有引用关系一致。

### 具体业务案例：工作流模板深拷贝

#### 业务背景

在审批系统或工作流系统里，一条流程可能不仅有顺序执行关系，还存在“跳转节点”“回退节点”“补偿节点”等额外引用。

#### 输入是什么

输入是一条流程节点链：`next` 表示正常顺序，`random` 表示额外引用关系。

#### 算法介入点

当系统要复制一份流程模板时，不能只复制节点值，还必须把所有内部引用关系同步迁移到新模板里。

#### 输出是什么

输出是一条全新的流程链，结构一致，但不与原模板共享任何节点对象。

#### 价值是什么

它解决的是复杂引用结构的深拷贝问题，避免新旧对象之间互相污染。


In [ ]:
def build_random_list(spec: list[tuple[int, int | None]]) -> Node | None:
    if not spec:
        return None

    nodes = [Node(val) for val, _ in spec]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]

    for node, (_, random_index) in zip(nodes, spec):
        node.random = nodes[random_index] if random_index is not None else None

    return nodes[0]


def serialize_random_list(head: Node | None) -> list[tuple[int, int | None]]:
    nodes: list[Node] = []
    index_map: dict[Node, int] = {}
    cur = head

    while cur is not None:
        index_map[cur] = len(nodes)
        nodes.append(cur)
        cur = cur.next

    result: list[tuple[int, int | None]] = []
    for node in nodes:
        random_index = index_map[node.random] if node.random is not None else None
        result.append((node.val, random_index))
    return result


sample = [(7, None), (13, 0), (11, 1), (10, 2), (1, 0)]
head = build_random_list(sample)
copied = Solution().copyRandomList(head)

print('原链表序列化:', serialize_random_list(head))
print('复制链表序列化:', serialize_random_list(copied))
print('头节点是否为同一对象:', head is copied)


## 13. Demo 输出说明

- 两次序列化结果应该完全一致，说明 `val`、`next` 和 `random` 结构都复制正确。
- `head is copied` 应该是 `False`，说明返回的是独立的新链表，而不是原链表。
- 如果运行后原链表序列化结果仍然没变，也说明第三步拆链时旧链表恢复正确。


## 14. 一句话复盘

> 这题最关键的不是复制节点值，而是像 Java 原解那样把“旧节点到新节点的映射”藏进链表穿插结构里。
